# OCR tiếng Việt bằng PaddleOCR (tuỳ chọn)

Backend hiện chưa có nguồn tìm kiếm OCR; kết quả dùng để thử nghiệm.

- **Đầu vào:** `<keyframe-dir>/<video_id>/<shot_id>/<keyframe>.jpg`.
- **Đầu ra:** `/kaggle/working/ocr/<video_id>/<shot_id>/<keyframe>.json`
  `{video_id, shot_id, keyframe_id, full_text, texts: [{text, confidence, polygon}]}`.

Cài paddlepaddle-gpu mất vài phút. Bật GPU (T4 x2).

In [ ]:
!pip install -q "numpy<2"
!pip install -q --force-reinstall scikit-image==0.22.0 scipy==1.12.0
!pip install -q paddlepaddle-gpu paddleocr==2.8.1 opencv-python-headless

## Cấu hình
Ảnh được chia cho các máy theo khối liên tiếp sau khi sắp xếp đường dẫn.

In [ ]:
from pathlib import Path

KEYFRAME_DIR = Path("/kaggle/input/datasets/trncqun/aic26-keyframes/keyframes")
OUTPUT_DIR = Path("/kaggle/working/ocr")
LANG = "vi"
MIN_CONFIDENCE = 0.5

SHARD_INDEX = 1  # ← đổi theo máy: 1, 2, 3 hoặc 4
NUM_SHARDS = 4

## Chạy OCR
Ảnh lỗi không ghi file JSON, nên chạy lại cell sẽ thử lại đúng các ảnh đó.

In [ ]:
import json

from paddleocr import PaddleOCR
from tqdm.auto import tqdm


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


def parse_lines(result):
    """PaddleOCR returns [None] without text; some boxes come without a recognized string."""
    texts = []
    for line in (result[0] or []) if result else []:
        if len(line) < 2 or not isinstance(line[1], tuple | list) or len(line[1]) < 2:
            continue
        text, confidence = line[1][0], float(line[1][1])
        if text and confidence >= MIN_CONFIDENCE:
            texts.append({"text": text, "confidence": round(confidence, 4), "polygon": line[0]})
    return texts


# use_angle_cls xoay lại các dòng chữ bị nghiêng hoặc lộn ngược trước khi nhận dạng.
ocr = PaddleOCR(use_angle_cls=True, lang=LANG, enable_mkldnn=False)
images = select_shard(list(KEYFRAME_DIR.rglob("*.jpg")), SHARD_INDEX, NUM_SHARDS)
print(f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(images)} ảnh")

failed = 0
for image_path in tqdm(images, desc="OCR"):
    relative = image_path.relative_to(KEYFRAME_DIR)
    out_json = (OUTPUT_DIR / relative).with_suffix(".json")
    if out_json.exists():
        continue
    try:
        texts = parse_lines(ocr.ocr(str(image_path)))
    except Exception as exc:  # noqa: BLE001 - một ảnh lỗi không được dừng cả lô
        print(f"[LỖI] {image_path}: {exc}")
        failed += 1
        continue
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(
        json.dumps(
            {
                "video_id": relative.parts[0],
                "shot_id": relative.parts[1] if len(relative.parts) > 2 else "",
                "keyframe_id": image_path.stem,
                "full_text": " ".join(item["text"] for item in texts),
                "texts": texts,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
print(f"Xong; {failed} ảnh lỗi")

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/ocr/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")